# 72 ? Experimento TF-IDF + DBSCAN

Este notebook testa agrupamento de descri??es normalizadas de itens de supermercado usando TF-IDF e DBSCAN. A an?lise usa dist?ncia cosseno, n?o usa NCM nem r?tulos humanos, e trata ru?do como resultado expl?cito do algoritmo. A amostra configur?vel mant?m o custo do c?lculo de vizinhan?a controlado. Execute de cima para baixo em uma sess?o limpa.


In [17]:
from datetime import datetime, timezone
from pathlib import Path
import logging

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import DBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score

from nofis_classifier.schema import DESCRIPTION_NORMALIZED, FREQUENCY

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("notebook72")
pd.set_option("display.max_colwidth", 140)


## 1. Parametros

`eps` e o raio maximo de vizinhanca na distancia cosseno; `min_samples` e o numero minimo de pontos para formar uma regiao densa. A escolha e sensivel a escala de `eps`, entao a grade inicial e exploratoria. Silhouette sera calculada somente nos itens nao classificados como ruido e nao representa qualidade semantica nem acuracia.

A analise cruza duas sementes de amostragem com dois analyzers. Para isolar os efeitos, cada analyzer usa a mesma amostra da sua semente. O analyzer `word` usa unigramas e bigramas, enquanto `char_wb` usa n-gramas de caracteres. A grade de `eps` e exploratoria para ambos; as distancias podem ter escalas diferentes.



In [18]:
DATASET_NAME = "items-notas-fiscais-ncms-supermercados-com-duplicatas-experimentos-texto"
PERIOD_LABEL = "202501-202504"
DATASET_RELATIVE_PATH = Path("data/processed/items-notas-fiscais-ncms-supermercados-com-duplicatas-experimentos-texto/items-notas-fiscais-ncms-supermercados-com-duplicatas-experimentos-texto-202501-202504.parquet")
DATASET_PATH = next(
    (root / DATASET_RELATIVE_PATH for root in (Path.cwd(), *Path.cwd().parents)
     if (root / DATASET_RELATIVE_PATH).is_file()),
    DATASET_RELATIVE_PATH,
)
if not DATASET_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset nao encontrado: {DATASET_RELATIVE_PATH}. "
        "Execute o notebook em uma pasta dentro do repositorio e confira data/processed/."
    )
MAX_DESCRIPTIONS = 20_000  # None usa todas; DBSCAN pode ficar caro em bases grandes.
SAMPLE_SEEDS = [42, 123]
ANALYZER_CONFIGS = {
    "char_wb": (3, 5),
    "word": (1, 2),
    "word": (1, 3),
}
MIN_DF = 2
MAX_FEATURES = 60_000
EPS_VALUES = [0.10, 0.15, 0.20, 0.25, 0.30, 0.50]
MIN_SAMPLES_VALUES = [5, 10, 20, 40, 100]
OUTPUT_ROOT = Path("reports/tfidf_dbscan")


## 2. Fun??es da an?lise

A prepara??o remove descri??es vazias, mant?m uma linha por descri??o e valida que a frequ?ncia total n?o foi alterada. Cada configura??o ? executada em uma c?pia l?gica da mesma matriz TF-IDF.


In [19]:
def prepare_items(frame):
    initial_frequency = frame[FREQUENCY].sum()
    prepared = frame[[DESCRIPTION_NORMALIZED, FREQUENCY]].copy()
    prepared[DESCRIPTION_NORMALIZED] = prepared[DESCRIPTION_NORMALIZED].astype("string").str.strip()
    prepared = prepared[prepared[DESCRIPTION_NORMALIZED].notna() & prepared[DESCRIPTION_NORMALIZED].ne("")]
    prepared = prepared[prepared[FREQUENCY] >= 1].copy()
    prepared = (
        prepared.groupby(DESCRIPTION_NORMALIZED, as_index=False, sort=False)[FREQUENCY]
        .sum()
    )
    if prepared[FREQUENCY].sum() != initial_frequency:
        raise ValueError("A prepara??o alterou a soma de frequ?ncia.")
    return prepared.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)

def summarize_clusters(items, labels):
    result = items.copy()
    result["cluster"] = labels
    result["ruido"] = result["cluster"].eq(-1)
    summary = (
        result.groupby("cluster", as_index=False)
        .agg(descricoes_unicas=(DESCRIPTION_NORMALIZED, "size"), frequencia_total=(FREQUENCY, "sum"))
        .sort_values("descricoes_unicas", ascending=False)
    )
    return result, summary

def score_configuration(matrix, labels):
    labels = np.asarray(labels)
    noise_mask = labels == -1
    clustered_labels = labels[~noise_mask]
    cluster_count = len(np.unique(clustered_labels))
    score = np.nan
    if cluster_count >= 2 and cluster_count < len(clustered_labels):
        score = float(silhouette_score(matrix[~noise_mask], clustered_labels, metric="cosine"))
    return {
        "cluster_count_excluding_noise": cluster_count,
        "noise_count": int(noise_mask.sum()),
        "noise_fraction": float(noise_mask.mean()),
        "silhouette_cosine_without_noise": score,
    }


## 3. Carregar e preparar dados

Origem: Parquet produzido pelos notebooks de preparacao de dados do projeto, com descricoes de supermercado, NCMs e duplicatas agregadas por descricao. A amostragem e feita uma vez por semente e reutilizada entre os analyzers daquela semente.


In [20]:
base_items = prepare_items(pd.read_parquet(DATASET_PATH))
logger.info("Descricoes disponiveis apos preparacao: %d", len(base_items))


2026-10-11 07:23:07,860 INFO Descricoes disponiveis apos preparacao: 70863


## 4. Vetorizar, ajustar DBSCAN e comparar parametros

Cada combinacao de semente e analyzer gera uma matriz TF-IDF propria. `metric="cosine"` permite trabalhar diretamente com a matriz esparsa. O ruido (`cluster = -1`) e excluido da silhouette e contabilizado separadamente; configuracoes com menos de dois clusters validos ficam sem silhouette. A matriz completa nao e convertida para densa.


In [21]:
records = []
labels_by_run = {}
items_by_run = {}

for sample_seed in SAMPLE_SEEDS:
    sampled_items = base_items
    if MAX_DESCRIPTIONS is not None and len(base_items) > MAX_DESCRIPTIONS:
        sampled_items = base_items.sample(n=MAX_DESCRIPTIONS, random_state=sample_seed)
    sampled_items = sampled_items.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)

    for analyzer, ngram_range in ANALYZER_CONFIGS.items():
        logger.info(
            "Vetorizando analyzer=%s, seed=%d, descricoes=%d",
            analyzer, sample_seed, len(sampled_items),
        )
        vectorizer = TfidfVectorizer(
            analyzer=analyzer, ngram_range=ngram_range, min_df=MIN_DF,
            max_features=MAX_FEATURES, sublinear_tf=True, norm="l2",
        )
        matrix = vectorizer.fit_transform(sampled_items[DESCRIPTION_NORMALIZED].astype(str))
        zero_vector_count = int((matrix.getnnz(axis=1) == 0).sum())
        logger.info(
            "Analyzer=%s seed=%d: %d termos, %d vetores zerados",
            analyzer, sample_seed, matrix.shape[1], zero_vector_count,
        )

        for eps in EPS_VALUES:
            for min_samples in MIN_SAMPLES_VALUES:
                run_id = f"seed{sample_seed}_{analyzer}_eps{eps:g}_min_samples{min_samples}"
                logger.info("Ajustando DBSCAN %s", run_id)
                model = DBSCAN(
                    eps=eps, min_samples=min_samples, metric="cosine",
                    algorithm="brute", n_jobs=-1,
                )
                model.fit(matrix, sample_weight=sampled_items[FREQUENCY].to_numpy(dtype=float))
                labels = model.labels_
                metrics = score_configuration(matrix, labels)
                records.append({
                    "run_id": run_id, "sample_seed": sample_seed,
                    "analyzer": analyzer, "ngram_range": str(ngram_range),
                    "eps": eps, "min_samples": min_samples,
                    "frequency_weighted": True, **metrics,
                })
                labels_by_run[run_id] = labels
                items_by_run[run_id] = sampled_items

results = pd.DataFrame(records).sort_values(
    ["silhouette_cosine_without_noise", "noise_fraction"],
    ascending=[False, True], na_position="last",
).reset_index(drop=True)
display(results)


2026-10-11 07:23:07,923 INFO Vetorizando analyzer=char_wb, seed=42, descricoes=20000
2026-10-11 07:23:12,327 INFO Analyzer=char_wb seed=42: 43949 termos, 0 vetores zerados
2026-10-11 07:23:12,330 INFO Ajustando DBSCAN seed42_char_wb_eps0.1_min_samples5
2026-10-11 07:23:21,854 INFO Ajustando DBSCAN seed42_char_wb_eps0.1_min_samples10
2026-10-11 07:23:31,859 INFO Ajustando DBSCAN seed42_char_wb_eps0.1_min_samples20
2026-10-11 07:23:40,212 INFO Ajustando DBSCAN seed42_char_wb_eps0.1_min_samples40
2026-10-11 07:23:49,765 INFO Ajustando DBSCAN seed42_char_wb_eps0.1_min_samples100
2026-10-11 07:23:59,856 INFO Ajustando DBSCAN seed42_char_wb_eps0.15_min_samples5
2026-10-11 07:24:11,966 INFO Ajustando DBSCAN seed42_char_wb_eps0.15_min_samples10
2026-10-11 07:24:25,874 INFO Ajustando DBSCAN seed42_char_wb_eps0.15_min_samples20
2026-10-11 07:24:37,079 INFO Ajustando DBSCAN seed42_char_wb_eps0.15_min_samples40
2026-10-11 07:24:47,666 INFO Ajustando DBSCAN seed42_char_wb_eps0.15_min_samples100
202

,run_id,sample_seed,analyzer,ngram_range,eps,min_samples,frequency_weighted,cluster_count_excluding_noise,noise_count,noise_fraction,silhouette_cosine_without_noise
0,seed42_word_eps0.2_min_samples100,42,word,"(1, 3)",0.20,100,True,78,19771,0.98855,0.690146
1,seed123_word_eps0.25_min_samples100,123,word,"(1, 3)",0.25,100,True,93,19644,0.98220,0.674636
2,seed42_char_wb_eps0.15_min_samples100,42,char_wb,"(3, 5)",0.15,100,True,83,19681,0.98405,0.674528
3,seed42_char_wb_eps0.1_min_samples100,42,char_wb,"(3, 5)",0.10,100,True,79,19813,0.99065,0.670212
4,seed42_word_eps0.15_min_samples100,42,word,"(1, 3)",0.15,100,True,76,19823,0.99115,0.669872
...,...,...,...,...,...,...,...,...,...,...,...
115,seed123_char_wb_eps0.5_min_samples20,123,char_wb,"(3, 5)",0.50,20,True,106,6380,0.31900,-0.074442
116,seed42_char_wb_eps0.5_min_samples10,42,char_wb,"(3, 5)",0.50,10,True,237,4903,0.24515,-0.099324
117,seed123_char_wb_eps0.5_min_samples10,123,char_wb,"(3, 5)",0.50,10,True,256,4967,0.24835,-0.116220
118,seed42_char_wb_eps0.5_min_samples5,42,char_wb,"(3, 5)",0.50,5,True,502,3726,0.18630,-0.127484


## 5. Revisar um resultado e salvar relat?rios

Escolha `SELECTED_RUN_ID` ap?s comparar quantidade de clusters, ru?do e silhouette. A tabela de resumo inclui `-1` como grupo de ru?do. Os arquivos s?o gravados em uma pasta com identificador UTC ?nico para preservar execu??es anteriores.


In [22]:
SELECTED_RUN_ID = results.iloc[0]["run_id"]
selected = results.set_index("run_id").loc[SELECTED_RUN_ID]
selected_items = items_by_run[SELECTED_RUN_ID]
selected_labels = labels_by_run[SELECTED_RUN_ID]
assignments, cluster_summary = summarize_clusters(selected_items, selected_labels)
display(cluster_summary)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_dir = OUTPUT_ROOT / f"{PERIOD_LABEL}_{RUN_ID}"
output_dir.mkdir(parents=True, exist_ok=False)
results.to_csv(output_dir / "metrics.csv", index=False)
assignments.to_csv(output_dir / "assignments.csv", index=False, encoding="utf-8-sig")
cluster_summary.to_csv(output_dir / "cluster_summary.csv", index=False, encoding="utf-8-sig")
run_metadata = {
    "dataset_name": DATASET_NAME, "period": PERIOD_LABEL,
    "dataset_path": str(DATASET_PATH), "max_descriptions": MAX_DESCRIPTIONS,
    "sample_seeds": SAMPLE_SEEDS, "analyzer_configs": {
        analyzer: list(ngram_range) for analyzer, ngram_range in ANALYZER_CONFIGS.items()
    },
    "min_df": MIN_DF, "max_features": MAX_FEATURES,
    "selected_run_id": SELECTED_RUN_ID,
    "eps_values": EPS_VALUES, "min_samples_values": MIN_SAMPLES_VALUES,
}
(output_dir / "run.json").write_text(
    __import__("json").dumps(run_metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)
logger.info("Relatorios salvos em %s", output_dir)


,cluster,descricoes_unicas,frequencia_total
0,-1,19771,63294
1,0,14,120
15,14,12,199
18,17,12,176
58,57,10,439
...,...,...,...
72,71,1,204
71,70,1,145
75,74,1,187
76,75,1,114


2026-10-11 07:40:17,351 INFO Relatorios salvos em reports\tfidf_dbscan\202501-202504_20261011T104017Z


### Interpreta??o

- `noise_fraction` alta significa que muitas descri??es n?o entraram em regi?es densas para os par?metros escolhidos.
- Poucos clusters ou um cluster dominante podem indicar que `eps` est? grande; quase tudo como ru?do pode indicar `eps` pequeno ou `min_samples` alto.
- Silhouette mais alta deve ser lida junto da fra??o de ru?do: uma pontua??o calculada em poucos itens pode ser enganosa.
- DBSCAN n?o atribui automaticamente novos produtos a clusters ap?s o ajuste e n?o define categorias sem revis?o humana.
